# BiLSTM 

In [19]:
# Bidirectional LSTM for Hinglish Offensive-Content Classification (Keras)

import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import fasttext

In [23]:
!git clone https://github.com/USERNAME/REPOSITORY.git /content/HINSAFE

fatal: destination path '/content/HINSAFE' already exists and is not an empty directory.


In [24]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/HINSAFE")
sys.path.insert(0, str(PROJECT_ROOT))

print("Project exists:", PROJECT_ROOT.exists())
print("src exists:", (PROJECT_ROOT / "src").exists())

Project exists: True
src exists: True


In [25]:
from src.preprocessing import basic_clean
from src.evaluate import evaluate_model

In [26]:
pd.set_option("display.max_colwidth", 100)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [35]:
# File paths

TRAIN_PATH = PROJECT_ROOT / "datasets" / "processed" / "train_split.csv"
VAL_PATH = PROJECT_ROOT / "datasets" / "processed" / "val_split.csv"
TEST_PATH = PROJECT_ROOT / "datasets" / "processed" / "test_split.csv"

FASTTEXT_MODEL_PATH = PROJECT_ROOT / "embeddings" / "cc.hi.300.bin"

MODEL_PATH = PROJECT_ROOT / "saved_models" / "bilstm_keras.h5"
TOKENIZER_PATH = PROJECT_ROOT / "saved_models" / "bilstm_tokenizer.pkl"

RESULTS_PATH = PROJECT_ROOT / "results" / "model_comparison.csv"
CONFUSION_MATRIX_PATH = PROJECT_ROOT / "results" / "confusion_matrices" / "bilstm.png"
TRAINING_HISTORY_PATH = PROJECT_ROOT / "results" / "bilstm_training_history.png"

(PROJECT_ROOT / "saved_models").mkdir(exist_ok=True)
(PROJECT_ROOT / "results" / "confusion_matrices").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "embeddings").mkdir(exist_ok=True)

print("Train file exists:", TRAIN_PATH.exists())
print("Validation file exists:", VAL_PATH.exists())
print("Test file exists:", TEST_PATH.exists())

Train file exists: True
Validation file exists: True
Test file exists: True


In [36]:
# Model settings 
MAX_VOCAB_SIZE = 20000     # only keep the top 20,000 most frequent words
MAX_SEQUENCE_LENGTH = 50   # pad/truncate every comment to 50 tokens
EMBEDDING_DIM = 300        # FastText vectors are 300-dimensional
LSTM_UNITS = 128
BATCH_SIZE = 32
EPOCHS = 10

In [37]:

# Load datasets

try:
    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VAL_PATH)
    test_df = pd.read_csv(TEST_PATH)

    print("Datasets loaded successfully!\n")

    print("Training set:", train_df.shape)
    print("Validation set:", val_df.shape)
    print("Test set:", test_df.shape)

except FileNotFoundError as e:
    print("Error: Could not find one of the dataset files.")
    print("Please check the file paths.")
    print(e)

Datasets loaded successfully!

Training set: (4288, 2)
Validation set: (920, 2)
Test set: (920, 2)


In [38]:
required_columns = ["text", "label"]

for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        print(f"{name} set is missing: {missing_columns}")
    else:
        print(f"{name} set: OK")

print("\nExample training data:")
display(train_df[["text", "label"]].head())

Training set: OK
Validation set: OK
Test set: OK

Example training data:


,text,label
0,mama ke dabre se bahir a gaye ho kya baat ha chote bacha ka rape howa tha qasoor me baad me pata...,1
1,chinta kt kro beta madhur tmko tmri bhakti ke lie rajya sabha mn entry mil jaegi bo bhi rape ke ...,0
2,modi ka to pata nahi par aj vi delhi me rape ro raha hai tum log sirf modi modi modi ka mala jap...,0
3,teri gaand ka business,1
4,aur hum par terrorism ka label lagane wale duniya ache tarah jante hai aur tu bhe india is the m...,1


In [39]:
# Tokenization 
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

# convert text to sequences of integers 
train_sequences = tokenizer.texts_to_sequences(train_df["text"])
val_sequences = tokenizer.texts_to_sequences(val_df["text"])


In [43]:
# Padding sequences to ensure uniform length

X_train = pad_sequences(
    train_sequences, maxlen=MAX_SEQUENCE_LENGTH, 
    padding="post", truncating="post")
X_val = pad_sequences(
    val_sequences, maxlen=MAX_SEQUENCE_LENGTH, 
    padding="post", truncating="post")

y_train = train_df["label"].values
y_val = val_df["label"].values

In [44]:

print("Vocabulary size found:", len(tokenizer.word_index))
print("Training sequence shape:", X_train.shape)
print("Validation sequence shape:", X_val.shape)

Vocabulary size found: 11621
Training sequence shape: (4288, 50)
Validation sequence shape: (920, 50)


In [45]:
# See what one comment looks like after tokenizing and padding
print("\nExample - original text:", train_df["text"].iloc[0])
print("Example - as sequence:", train_sequences[0])
print("Example - padded:", X_train[0])


Example - original text: mama ke dabre se bahir a gaye ho kya baat ha chote bacha ka rape howa tha qasoor me baad me pata chala ke tum b un ke sath the ye ha hidmat
Example - as sequence: [2806, 7, 4374, 9, 1051, 265, 152, 11, 24, 56, 87, 1052, 409, 5, 3, 1671, 35, 2078, 15, 195, 15, 162, 362, 7, 41, 127, 232, 7, 96, 98, 22, 87, 4375]
Example - padded: [2806    7 4374    9 1051  265  152   11   24   56   87 1052  409    5
    3 1671   35 2078   15  195   15  162  362    7   41  127  232    7
   96   98   22   87 4375    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]
